In [76]:
import json

In [77]:
image_dir = "E:/Research/simplify-me/simplify_me_dataset/images"
meta_path = "E:/Research/simplify-me/simplify_me_dataset/meta.json"
environment = 'local'

REGISTRY = {
    'local': {
        'image_dir': "E:/Research/simplify-me/simplify_me_dataset/images",
        'meta_path': "E:/Research/simplify-me/simplify_me_dataset/meta.json",
        'environment': 'local',
    },
    'cc': {
        'image_dir': "/home/pranon/scratch/def-tahmedge/simplify-me-dataset/simplify-me/compressed_images",
        'meta_path': "E:/Research/simplify-me/simplify_me_dataset/meta.json",
        'environment': 'cc',
    }
}

In [78]:
def load_data():
    with open(meta_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [94]:
def get_and_preprocess_data(data, split, env, ds_size):
    processed_data = []

    data = data[split]

    if split == 'train':
        data = sorted(
            data,
            key=lambda x: float(x['dpo']['sle_delta']),
            reverse=True
        )

        if ds_size != -1:
            data = data[:ds_size]
        print(split, env, len(data), data[-1]['dpo']['sle_delta'])

    for item in data:
        processed_data.append({
            'id': str(item['id']),
            'benchmark': item['benchmark'],
            'input': "<image>",
            'images': [f"{REGISTRY[env]['image_dir']}/{item['benchmark']}/{item['image']['file_name']}"],
            'instruction': 'Describe the image in simple terms',
            'chosen': item['dpo']['accepted']['caption'],
            'rejected': item['dpo']['rejected']['caption'],
        })

    filename = f'../data/{split}-{env}{'' if split != 'train' else '-full' if ds_size == -1 else '-'+str(ds_size)}.json'

    with open(filename, 'w', encoding='utf-8') as fp:
        json.dump(processed_data, fp, indent=4, ensure_ascii=False)

    return filename

In [95]:
def get_dataset_info(env, split, filename, ds_size):
    return {
        f'sm-{env}-{split}{'' if split != 'train' else '-full' if ds_size == -1 else '-'+str(ds_size)}': {
            "file_name": filename,
            "ranking": True,
            "columns": {
                "prompt": "instruction",
                "query": "input",
                "chosen": "chosen",
                "rejected": "rejected",
                "images": "images"
            }
        },
    }

In [96]:
raw_data = load_data()

dataset_info = {}
max_size = [-1, 10000, 5000, 1000]

for key in REGISTRY.keys():
    for sz in max_size:
        fn = get_and_preprocess_data(raw_data, 'train', key, sz)
        ds_info = get_dataset_info(key, 'train', fn, sz)

        dataset_info.update(ds_info)

        fn = get_and_preprocess_data(raw_data, 'test', key, sz)
        ds_info = get_dataset_info(key, 'test', fn, sz)
        dataset_info.update(ds_info)

with open('../data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=4, ensure_ascii=False)

train local 92174 2.000012755393982
train local 10000 3.366675615310669
train local 5000 3.6479693055152893
train local 1000 4.210689380764961
train cc 92174 2.000012755393982
train cc 10000 3.366675615310669
train cc 5000 3.6479693055152893
train cc 1000 4.210689380764961
